In [1]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import time


In [2]:
df = pd.read_csv(r'D:\Downloads\scenario_manifest.csv')

In [3]:

# Assuming your dataframe is 'df' and the column with these classes is 'taxonomy'
normal_mask = df['scenario_id'].str.startswith('normal_')

# Sample exactly 500 from each 'normal' class (stratified sampling)
df_normal_sampled = (
    df[normal_mask]
    .groupby('scenario_id', group_keys=False)
    .apply(lambda x: x.sample(n=500, random_state=42))
)

# Keep the non-normal classes as they are
df_others = df[~normal_mask]

# Combine and shuffle the final dataset
df_balanced = pd.concat([df_normal_sampled, df_others]).sample(frac=1, random_state=42).reset_index(drop=True)

C:\Users\bassa\AppData\Local\Temp\ipykernel_7104\2136101372.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=500, random_state=42))


In [4]:
print("\n=== CATEGORICAL VALUES ===")
lis = df_balanced.select_dtypes(include=['object']).columns[1:]
for col in lis:
        print(f"\n{col}: {df_balanced[col].nunique()} unique values")
        print(df_balanced[col].value_counts().head(21))


=== CATEGORICAL VALUES ===

scenario_id: 21 unique values
scenario_id
raf_alg_mismatch                     500
model_gray_rsa_mimicry               500
anomaly_pqc_over_tls12               500
normal_mlkem768_mldsa65_cert         500
net_high_loss_pqc                    500
normal_mlkem768_mldsa87_cert         500
raf_downgrade_overt                  500
anomaly_kem_mismatch                 500
net_high_jitter_classic              500
normal_mlkem1024_mldsa87_cert        500
normal_mlkem768_rsa_cert             500
threat_data_exfil                    500
normal_X25519_ecdsa_p256_cert        500
net_high_latency_pqc                 500
normal_X25519_rsa_cert               500
normal_X25519_rsa3072_cert           500
normal_mlkem1024_rsa_cert            500
normal_mlkem1024_mldsa65_cert        500
normal_X25519_ecdsa_p384_cert        500
anomaly_hybrid_pki                   500
raf_robustness_malformed_keyshare     10
Name: count, dtype: int64

taxonomy_class: 5 unique values
taxonomy_

In [5]:
temp = pd.read_csv(r'D:\Downloads\df_with_idx.csv')

In [6]:
temp = temp.drop(columns=['Unnamed: 0'])

In [7]:
df_balanced['ID'] = df_balanced['ID'].str.replace('.pcap', '', regex=False)

In [8]:
# The tilde (~) means "NOT". So this keeps rows where the ID is NOT in df
unmatched_df = temp[~temp['ID'].isin(df_balanced['ID'])]

In [9]:
# Option 1: Filter 'temp' to only keep rows with IDs that exist in 'df'
df = temp[temp['ID'].isin(df_balanced['ID'])]
splits = df['split'].str.lower().str.strip()
train_mask = (splits == 'train')
val_mask = (splits == 'val')
test_mask = (splits == 'test')

In [10]:
df = df.drop(columns=[ 'split', 'taxonomy', 'ID' ,'idx'])

In [11]:
# df.to_csv(r'D:\Downloads\df_short.csv')
lis = df.select_dtypes(include=['bool'])
for col in lis:
    df[col] = le.fit_transform(df[col])

In [63]:
# 1. LOAD DATA
# DATA_PATH = r'D:\Downloads\df_short.csv'
# df = pd.read_csv(DATA_PATH)

# 2. SEPARATE TARGET AND DROP METADATA (Keep 'subtype' for now!)
cols_to_drop = ['label']
for col in ['taxonomy', 'ID', 'Unnamed: 0']: 
    if col in df.columns:
        cols_to_drop.append(col)
        
X = df.drop(columns=cols_to_drop)
y = df['label'].astype('int32') 

# 3. SPLIT THE DATA STRATIFYING BY 'subtype'
# Step A: Split off 70% for Training, stratifying on X['subtype']
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=X['subtype']
)

# Step B: Split the remaining 30% in half to get 15% Val and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=X_temp['subtype']
)

# =======================================================
# --- NEW: DROP 'subtype' BEFORE ANY TRAINING OCCURS ---
# =======================================================
print("Dropping 'subtype' from feature sets after splitting...")
X_train = X_train.drop(columns=['subtype']).astype('float32')
X_val = X_val.drop(columns=['subtype']).astype('float32')
X_test = X_test.drop(columns=['subtype']).astype('float32')


# 4. CALCULATE SCALE_POS_WEIGHT FOR IMBALANCE
num_negatives = (y_train == 0).sum()
num_positives = (y_train == 1).sum()
scale_weight = num_negatives / num_positives if num_positives > 0 else 1


# =======================================================
# PHASE 1: INITIAL TRAINING TO FIND FEATURE IMPORTANCES
# =======================================================
print("\n--- PHASE 1: INITIAL TRAINING ---")
initial_xgb = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    scale_pos_weight=scale_weight,
    random_state=42,
    early_stopping_rounds=20
)

# We can run this silently (verbose=False) just to get the importances quickly
initial_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)


# =======================================================
# PHASE 2: DROP 0 IMPORTANCE FEATURES
# =======================================================
importances = initial_xgb.feature_importances_
feature_names = initial_xgb.feature_names_in_ 

zero_importance_mask = (importances == 0.0)
features_to_drop = feature_names[zero_importance_mask]

print(f"\nDropping {len(features_to_drop)} features with 0 importance...")

X_train_reduced = X_train.drop(columns=features_to_drop)
X_val_reduced = X_val.drop(columns=features_to_drop)
X_test_reduced = X_test.drop(columns=features_to_drop)

print(f"Old feature count: {X_train.shape[1]}")
print(f"New feature count: {X_train_reduced.shape[1]}\n")


# =======================================================
# PHASE 3: FINAL TRAINING ON REDUCED DATASET
# =======================================================
print("--- PHASE 2: FINAL TRAINING ---")
final_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    scale_pos_weight=scale_weight,
    random_state=42,
    early_stopping_rounds=50 
)

print(f"Training on {len(X_train_reduced)} samples...")
print(f"Validating on {len(X_val_reduced)} samples...")
print(f"Testing on {len(X_test_reduced)} samples...\n")

start = time.perf_counter()

final_xgb.fit(
    X_train_reduced,  
    y_train,
    eval_set=[(X_val_reduced, y_val)],
    verbose=100 
)

end = time.perf_counter()
training_time = end - start
print(f'\nFinal Model Training time: {training_time:.10f} seconds')

Dropping 'subtype' from feature sets after splitting...

--- PHASE 1: INITIAL TRAINING ---

Dropping 24 features with 0 importance...
Old feature count: 32
New feature count: 8

--- PHASE 2: FINAL TRAINING ---
Training on 7007 samples...
Validating on 1501 samples...
Testing on 1502 samples...

[0]	validation_0-logloss:0.66970
[100]	validation_0-logloss:0.16283
[200]	validation_0-logloss:0.14508
[281]	validation_0-logloss:0.14637

Final Model Training time: 0.1965263000 seconds


In [64]:
# 6. EVALUATE ON TEST SET
print("\nGenerating Classification Report on Test Data...")
start = time.perf_counter()

testing_time = end - start
y_pred = final_xgb.predict(X_test_reduced)
end = time.perf_counter()
testing_time = end - start
print(f'Testing time: {testing_time:.10f} seconds')
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))


Generating Classification Report on Test Data...
Testing time: 0.0046359000 seconds
              precision    recall  f1-score   support

     Class 0       0.90      0.93      0.91       750
     Class 1       0.93      0.89      0.91       752

    accuracy                           0.91      1502
   macro avg       0.91      0.91      0.91      1502
weighted avg       0.91      0.91      0.91      1502



In [61]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. PREPARE THE 30,000 ROW DATASET
cols_to_drop = ['label']
for col in ['split', 'taxonomy', 'ID', 'idx', 'subtype']: 
    if col in unmatched_df.columns:
        cols_to_drop.append(col)
        
X_unseen = unmatched_df.drop(columns=cols_to_drop)
y_unseen = unmatched_df['label']

# Convert to matching data types
X_unseen = X_unseen.astype('float32')
y_unseen = y_unseen.astype('int32') 

# =======================================================
# --- NEW: DROP 0 IMPORTANCE FEATURES FROM UNSEEN DATA ---
# =======================================================
# We use the 'features_to_drop' list created during the training script
print(f"Dropping the {len(features_to_drop)} zero-importance features from unseen data...")
X_unseen_reduced = X_unseen.drop(columns=features_to_drop, errors='ignore')

# 2. MAKE PREDICTIONS
# Make sure to use 'final_xgb' and 'X_unseen_reduced'
print(f"Testing model on {len(X_unseen_reduced)} unseen samples...\n")
y_pred_unseen = final_xgb.predict(X_unseen_reduced)

# 3. EVALUATE
print("--- Confusion Matrix ---")
cm = confusion_matrix(y_unseen, y_pred_unseen)
print(f"True Negatives (Correctly predicted as Normal): {cm[0][0]}")
if cm.shape[1] > 1:
    print(f"False Positives (Mistakenly predicted as Attack): {cm[0][1]}")
else:
    print("False Positives (Mistakenly predicted as Attack): 0")

print("\n--- Classification Report ---")
# We use zero_division=0 to prevent warnings since there are no Class 1 samples in the true labels
print(classification_report(y_unseen, y_pred_unseen, zero_division=0))

Dropping the 24 zero-importance features from unseen data...
Testing model on 30000 unseen samples...

--- Confusion Matrix ---
True Negatives (Correctly predicted as Normal): 27611
False Positives (Mistakenly predicted as Attack): 2389

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.92      0.96     30000
           1       0.00      0.00      0.00         0

    accuracy                           0.92     30000
   macro avg       0.50      0.46      0.48     30000
weighted avg       1.00      0.92      0.96     30000

